# דוח שבועי — כל המפעילים

שולף תכנון-מול-ביצוע לכל המפעילים בשבוע נתון, מחשב קנסות, ומפיק טבלאות + תרשימים.

> שימו לב: שליפה לכל המפעילים לשבוע שלם היא כבדה (מאות קווים). ה-cache המקומי מצמצם משמעותית ריצות חוזרות.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from stride_analysis.data.stride_client import StrideClient
from stride_analysis.analysis import calc_execution, calc_penalties, load_penalty_tables
from stride_analysis.analysis.penalties import penalties_by_operator
from stride_analysis.reports import generator
from stride_analysis.reports import charts

DATE_FROM, DATE_TO = '2026-05-24', '2026-05-30'
client = StrideClient()

## אופציה א׳ — טעינת תקציר קיים (מהיר, ללא רשת)
אם כבר קיים קובץ גלם, אפשר לדלג על השליפה.

In [ ]:
raw = pd.read_csv('../data/raw/_rides_execution_raw.csv')
summary = calc_penalties(
    calc_execution(raw, group_by=['operator_ref','operator_name','service_date']),
    load_penalty_tables())
summary.head()

## אופציה ב׳ — שליפה חיה לכל המפעילים (כבד)
בטלו את ההערה כדי לשלוף ישירות מה-API.

In [ ]:
# rides = client.fetch_rides(DATE_FROM, DATE_TO, operator_ref=None, progress=True)
# summary = calc_penalties(
#     calc_execution(rides, group_by=['operator_ref','operator_name','service_date']),
#     load_penalty_tables())

## דירוג מפעילים לפי קנס

In [ ]:
by_op = penalties_by_operator(summary)
by_op

## תרשימים

In [ ]:
charts.create_bar(by_op, label_col='operator_name', value_col='penalty_nis',
                  title='קנס אי-ביצוע לפי מפעיל (₪)')
charts.create_heatmap(summary, index_col='operator_name', column_col='service_date',
                      value_col='missing_pct', title='% אי-ביצוע לפי מפעיל ויום')
charts.create_timeline(summary, title='% אי-ביצוע לאורך השבוע')

## דוח עברית מלא

In [ ]:
from IPython.display import Markdown
md = generator.generate_report(summary, date_from=DATE_FROM, date_to=DATE_TO,
                               title='כלל המפעילים', daily=summary,
                               out_path='../data/output/weekly_all_operators.md')
Markdown(md)